<a href="https://colab.research.google.com/github/parthdabhi5195/YT-AI/blob/main/YT_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os

DATA_PATH = "/Users/parth/Desktop/Youtube/YT AI/Synthetic Training Data/story-pipeline v3/data/clean/final/stories.jsonl"
OUT_DIR = "slm_out"
LIMIT = 50000


In [5]:
!pip3 install tiktoken

In [6]:
!pip3 install datasets

In [7]:
!pip3 install -U datasets

## STEP 1: Importing Dataset


In [8]:
import json, random

stories = [json.loads(line)["story_text"] for line in open(DATA_PATH)]


In [9]:
print(stories[:2])

["The union hall hummed with the usual preamble to the weekly meeting, a comforting din I'd grown accustomed to, until a voice cut through the noise, calling my name. I looked up, my stomach clenching, to see Eddie, our union president, standing at the podium, a printed hotel receipt clutched in his hand, his gaze fixed solely on me.\n\nHe announced that a formal complaint had been filed, accusing me of misusing union funds, citing a discreet hotel stay. The accusation hung in the air, heavy and suffocating, threatening everything I’d worked for within the union. Patrick, my brother, a journalist who’d been covering our local for months, watched from the back, his expression unreadable.\n\nEddie led me to a small office, a makeshift interrogation room where Diane, our union’s auditor, sat waiting, her face stern. She questioned me about the receipt, about the hotel, about the money. My mind raced, a frantic internal monologue of denial and dawning fear. I was trapped, my reputation and

In [10]:

from datasets import load_dataset, Features, Value, Sequence

# needed due to schema change partway through file. Changes starts at line 1433252 in stories.jsonl
features = Features({
    "id": Value("string"),
    "story_text": Value("large_string"),
    "word_count": Value("int64"),
    "prompt_sha256": Value("string"),
    "source_template_id": Value("string"),
    "chunk": Value("string"),
    "setting": Value("string"),
    "occupation": Value("string"),
    "relationship_dynamic": Value("string"),
    "incident_seed": Value("string"),
    "hook_style": Value("string"),
    "character_names": Sequence(Value("string")),
    "source_title": Value("string"),
    "source_channel": Value("string"),
    "source_views": Value("int64"),
    "salvaged_from": Sequence(Value("string")),   # missing in the first 1.43M rows
})

ds = load_dataset("json", data_files=DATA_PATH, split="train", features=features)

if LIMIT:
    ds = ds.select(range(LIMIT))

ds = ds.train_test_split(test_size=0.01, seed=42, shuffle=True)
ds["val"] = ds.pop("test")      # two splits: train and val

print(ds)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'story_text', 'word_count', 'prompt_sha256', 'source_template_id', 'chunk', 'setting', 'occupation', 'relationship_dynamic', 'incident_seed', 'hook_style', 'character_names', 'source_title', 'source_channel', 'source_views', 'salvaged_from'],
        num_rows: 49500
    })
    val: Dataset({
        features: ['id', 'story_text', 'word_count', 'prompt_sha256', 'source_template_id', 'chunk', 'setting', 'occupation', 'relationship_dynamic', 'incident_seed', 'hook_style', 'character_names', 'source_title', 'source_channel', 'source_views', 'salvaged_from'],
        num_rows: 500
    })
})


## STEP 2: Tokenize the Dataset

(1) Tokenize the story dataset into tokenIDs

(2) Append <|endoftext|> after each story

(3) Create a file called "train.bin", "val.bin", "test.bin" to store all tokenIDs from the entire dataset. This is to increase speed (byte files don't need to parse characters like they have to in JSON), decrease storage size, and allow memory-mapping (letting us work with files larger than our RAM size by retrieving only active chunks).
(4) Make sure tokenIDs are stored to disk rather than RAM.

In [11]:
import tiktoken
import numpy as np
from tqdm.auto import tqdm # progress bar

enc = tiktoken.get_encoding("gpt2")
eot_id = enc.eot_token # <|endoftext|>
vocab_size = 50257 # from GPT2

def process(example):
    ids = enc.encode_ordinary(example['story_text']) # encode_ordinary ignores special tokens
    ids.append(eot_id)
    out = {'ids' : ids, 'len' : len(ids)} # {'ids' : list, len : int}
    return out

# Parallel Tokenization
if not os.path.exists(f"{OUT_DIR}/train.bin"):
    # ds.map() writes tokenIds to Array cache on disk, not RAM
    tokenized = ds.map (
        process,
        remove_columns=ds['train'].column_names, # remove all columns from output
        desc="tokenizing the splits", #tqdm labelled display
        num_proc=8 # applies the process function on every row in dataset across 8 threads
    )

    for split, dset in tokenized.items():
        # setting to uint64 to optimize storage. GPT2 vocab size is 50,257 so, 2^16 (65,535), every tokenID fits comfortably within this range
        arr_len = np.sum(dset['len'], dtype=np.uint64) # adding up the token counts in all rows of particular split to calculate exact number of tokens needed for array
        filename = f'{split}.bin'
        dtype = np.uint16 # stores tokenIDs as unsigned 16-bit values. Halves disk space compared to 32-bit integers
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,)) # allocates arr_len size memory mapped array in disk rather than in RAM
        total_batches = min(1024, len(dset)) # a split with fewer rows than shards can't be sharded. Used for safety with the limit flag

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # batch samples together for faster write
            batch = dset.shard(num_shards = total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids']) # flatten a list of individual stories into a 1D sequence since binary file is 1D
            # write into memmap
            arr[idx : idx + len(arr_batch)] = arr_batch # copy entire flattened block
            idx += len(arr_batch)
        arr.flush() # write to disk after all batches are processed

writing val.bin: 100%|██████████| 500/500 [00:00<00:00, 2150.07it/s]


In [12]:
# what encoding looks like
tokens = enc.encode(stories[1])
individual_tokens = [enc.decode([uid]) for uid in tokens]

print(tokens)
print("Split Text Chunks:", individual_tokens)

[1, 1639, 460, 470, 466, 428, 11, 314, 805, 72, 2474, 554, 25928, 35064, 988, 276, 11, 607, 1986, 542, 9741, 13, 220, 314, 6204, 612, 11, 19987, 11, 379, 616, 898, 10614, 16307, 11, 554, 25928, 6451, 287, 616, 1986, 13, 220, 1375, 13176, 379, 262, 35833, 21108, 314, 373, 4769, 11, 407, 329, 502, 11, 475, 329, 262, 27805, 314, 373, 546, 284, 1577, 13, 198, 198, 3347, 373, 3375, 546, 1223, 314, 1549, 13519, 1760, 11, 257, 29355, 11, 257, 6486, 326, 23273, 262, 48977, 8137, 13, 3244, 11, 673, 14678, 2004, 11, 2111, 284, 43388, 262, 21822, 422, 616, 1021, 11, 22187, 546, 703, 314, 2492, 470, 508, 314, 4752, 284, 307, 11, 703, 314, 373, 257, 7394, 13, 383, 13004, 11, 19632, 11, 10764, 1022, 514, 11, 788, 262, 14359, 4706, 11, 17799, 11, 13999, 625, 13, 198, 198, 464, 2647, 5025, 13, 317, 4791, 704, 4315, 7342, 13, 17799, 34676, 329, 2324, 11, 290, 2582, 734, 3790, 547, 13885, 11, 511, 14700, 18288, 13, 554, 25928, 11, 991, 275, 47883, 14227, 546, 502, 11, 546, 617, 3200, 2106, 11, 4030, 106

In [16]:
with open(f"{'/Users/parth/Desktop/Youtube/YT AI/YT AI SLM'}/train.bin", "rb") as f:
    data = f.read(50)

print(list(data))
print(enc.decode(data))

[40, 0, 184, 5, 177, 11, 11, 87, 11, 0, 104, 2, 31, 69, 11, 0, 193, 9, 246, 1, 99, 1, 1, 1, 252, 179, 11, 0, 1, 1, 62, 5, 169, 87, 148, 8, 30, 1, 119, 132, 11, 0, 104, 2, 37, 66, 13, 0, 220, 0]
I!�&�,,x,!�#@f,!*�"�"""��,!""_&�x�)?"��,!�#Fc.! !


## STEP 3: Creating Input-Output Batches for Dataset

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda" # nvidia gpu 
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps" # apple silicon
else:
    device = "cpu" # sadly cpu 

device_type = device

def get_batch(split):
    data = np.memmap(f"{OUT_DIR}/train.bin", dtype=np.uint16, mode="r") # reading from disk without loading entire dataset into RAM
    ix = torch.randint(len(data) - block_size - 1, (batch_size,)) # 1D tensor that randomly selects indices within the maximum index
    x = torch.stack([torch.from_numpy((data[i : i + batch_size]).astype(np.int64)) for i in ix]) # assemble current input batch. embedding layers strictly require int64. each slice is a numpy array of type int64
    y = torch.stack([torch.from_numpy((data[i + 1 : i + batch_size + 1]).astype(np.int64)) for i in ix]) # assemble associated correct answer
    if device_type == "cuda":
        # move tensors x, y from CPU memory to GPU memory and prevent OS from swapping out to disk (locked into RAM)
        # non_blocking creates asynchronous calls so CPU doesn't wait for GPU to finish processing and can do things like next batch preparation
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        # if using CPU, default to normal config
        x, y = x.to(device), y.to(device)

    return x, y

print(f"device: {device}")

device: mps


## STEP 4: Defining SLM architecture

In [24]:
import torch # base PyTorch library
import torch.nn as nn # neural network module (ex. nn.Linear, nn.Embedding)
import torch.nn.functional as F # F.layer_norm, F.softmax, F.cross_entropy
import math # square root scaling
from dataclasses import dataclass # structured data containers
import numpy as np 
from tqdm.auto import tqdm # progress bar generator
from contextlib import nullcontext # placeholder context manager
import os # filesystem operations

# Use LayerNorm instead of BatchNorm. LayerNorm is used in NLP when sentences come in variable length sequences.
# [5, 2, 7, 4, 1, 3]
# [8, 1, 6, X, X, X] why average over dimension 0 when columns near the end are mostly padding?
# [3, 9, X, X, X, X]
# [6, 4, 2, 7, 5, X]
class LayerNorm(nn.Module): # override nn.Module's __init__() and forward()
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim)) # gamma
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None # beta
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5) # gamma * (standarized feature row) + beta

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0 # ensures embedding dimension cleanly divides across all heads of attention
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_head, bias=config.bias) # (768, 3 * 768). 768 inputs, and 3x to later split Q, K, V
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias) # (768, 768) linear layer that mixes head outputs back into residual stream
        # takes percentage of random elements in attention affinity matrix (QK / sqrt(dk)) and sets to 0 during training (not inference)
        self.attn_dropout = nn.Dropout(config.dropout)# prevents model from relying heavily on single token-to-token relationships. Forces tokens to learn features from other tokens. Prevents overfitting
        # takes output vector y (after linear projection) but before it gets added to residual stream, and randomly 0s channel dimensions
        self.resid_droupout = nn.Dropout(config.dropout) # encourages model to not heavily rely on entire attention output and preserve more information through residual path
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        # prevents writing large (T, T) matrix into slow GPU VRAM. Rather, it creates smaller blocks inside fast SRAM
        self.flash = hasattr(F, 'scaled_dot_product_attention') # True or False flag to check if user's version supports FlashAttention (speeds attention while using less GPU memory)
        # create masked attention manually if not self.flash
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # x will be 3D
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2) # slices the 3 * C projection into individual Q, K, V tensors (B, T, C) 

        # reshape each tensor to (B, n_head, T, dk) to align each attention head as independent matrix batch
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            # faster way to implement the else condition
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else: 
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1))) # (B, n, T, dk) @ (B, n, dk, T) = (B, n, T, T)
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf')) # overwrites future tokens with -inf (necessary for softmax)
            att = F.softmax(att, dim=-1) # (B, n, T, T)
            att = self.attn_dropout(att) # (B, n, T, T)
            y = att @ v # (B, n, T, T) @ (B, n, T, dk) = (B, n, T, dk)
            return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias) # fully connected linear layer to 4x C dimension to facilitate deeper interactions
        self.gelu = nn.GELU() # Gaussian Error Linear Unit (nonlinearity like RELU or tanh)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias) # project from 4C into original C, with the assumption the model learned something new
        self.dropout = nn.Dropout(config.dropout) # prevent overfitting (model can't memorize training data)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x)))) # 4C projection on x, then perform nonlinearity, then bring back to original dimension, then perform dropout

class Block(nn.Module):
    # In accordance with nanoGPT architecture
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    # residual skip connection
    def forward(self, x):
         x += self.attn(self.ln1(x))
         x += self.mlp(self.ln2(x))
         return x

@dataclass # bundling all parameters into a single object
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

# High-level module that bundles together entire YT-AI transformer model
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size, config.n_embd), # word token embedding (maps integer token to respective higher dimensional vector)
        wpe = nn.Embedding(config.vocab_size, config.n_embd), # word position embedding (so model knows sequence of words)
        drop= nn.Dropout(config.dropout),
        h = nn.ModuleList([Block(config) for _ in range(config.n_layers)]),
        ln_f = LayerNorm(config.n_embd, config.bias) # final LayerNorm before projecting to logits
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) # linear projection to generate raw next-token prediction logits
        # prevents model from wasting capacity learning symmetrical input/output mappings. transposes the matrices internally
        self.transformer.wte.weight = self.lm_head.weight # points output projection matrix to the same memory and weights as the word token embedding.

        self.apply(self._init_weights) # recursively iterate through each submodule and initialize the weights of each module
        for pn, p in self.named_parameters(): # for parameter name, parameter tensor in interator of all model parameters
            if (pn.endswith('c_proj.weight')): # targetting the weights at the end of a residual connection
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))
                """
                Residual connections can compound variance if not checked. This can create activation saturation in GELU
                In a Gaussian Distribution, when you add by a constant, variance is unaffected.
                However, if you add by a variable, where each data point is added a variable number, variance can increase since extreme data points can compound.
                This is the case for our residual connections. So, we need to use scaling factors to bring variance back to 1, which is 1 / sqrt(2 * # of layers)
                """

    def _init_weights(self, module): # custom weight initialization to prevent vanishing gradients and exploding activations
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02) # start with a really low std to prevent saturation in GELU
            if module.bias is not None:
                nn.init.zeros_(module.bias) # so each next token predictions have equal probability at initialization
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean= 0.0, std= 0.02)

    # journey that data takes through the network (starts at text tokens, flows through each layer, and finally ends at next token prediction)
    def forward(self, idx, targets=None): # idx is input tensor containing token IDs (B, T)
        device = idx.device
        b, t  = idx.size() # (B, T)
        assert t <= self.config.block_size # if true, then continue. if false, raise error
        pos = torch.arange(0, t, dtype=torch.int64, device=device) # tensor [0, 1, 2, t-1] in CPU or GPU. so model sees "cat runs from dog" and "dog runs from cat" differently.
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos) # converts positions into vectors
        x = self.transformer.drop(tok_emb + pos_emb) # apply a drop to prevent overfitting

        for block in self.transformer.h:
            x = block(x) # loops through layernorm and attention mechanism

        x = self.transformer.ln_f(x) # apply final layer norm before making predictions

        if targets is not None: # training (targets) or inference (without targets)
            logits = self.lm_head(x) # (B, T, V), V = vocab_size
            # combines nn.LogSoftmax() and nn.NLLLoss()
            # when we pass in a matrix (B * T, V), are we going into every single row and calculating the -log(p), where p is the probability from correct next token (index in targets), and then averaging every -log(p) in B * T rows to get final loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1) # view logits as (B * T, V) and ground truths as (B * T, 1). In addition, skip any ground truths equal to -1
            return logits, loss
        else: # inference stage
            """
            grabs only the last token.
            then passes into lm_head which is a simple linear layer that takes in C for each B element.
            in production environments like Gemini, the model isn't just serving a single person's prompt at a time,
            it handles multiple independent user prompts at once. This is why we have the B dimension, so each
            prompt of T tokens can be processed at a single instance.
            but in our case, B dimension will just be 1 since we will be prompting one at a time.
            """
            logits = self.lm_head(x[:, [-1], :]) # takes (B, T, C) -> (B, 1, C), where 1 is the last token -> (B, 1, V)
            return logits, None # return None since we can't compute loss during inference
        
    
    @torch.no_grad() # disables gradient tracking during inference
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None): 
        """
        idx (B, T), 
        max_new_tokens (max token you want model to generate), 
        temperature (sampling randomness), 
        top_k (integer restricting sampling pool to top K most probable candidate tokens)

        Generating tokens given a conditional sequence of tokens
        """
        for _ in range(max_new_tokens):
            # as loop runs, idx grows longer. create a sliding window to keep context window with most recent generated token
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:] # negative slice to say "start from the end and get block_size back"
            logits, _ = self(idx_cond) # calls forward method on with current context window and returns (B, 1, V)
            # -1 mean drop dimension entirely, -1: means preserve it
            # Larger temp flattens out the ratio (making more random). Smaller temp sharpens distribution. after division, logits are exponentiated, which is why this happens (exponentiaion doesn't scale linearly)
            logits = logits[:, -1, :] / temperature # (B, V) / temperature. 
            if top_k is not None: # top_k prevents model from predicting gibberish words
                # find top-k largest logit in each row. safety of min() if top-k is larger than vocab_size
                v, _ = torch.topk(logits, min(top_k, logits.size(-1))) # (B, K)
                logits[logits < v[:, -1:]] = float('-inf') # create a boolean mask for every logit in vocab_size that is less than lowest top_k
            probs = F.softmax(logits, dim=-1) #  softmax (exponentiate and normalize) over V, where probs is (B, V) and B is typically 1 for 1 prompt
            idx_next = torch.multinomial(probs, num_samples=1) # sample the next probable token
            idx = torch.cat((idx, idx_next), dim=1) # append to the input token sequence
        return idx


In [ ]:
# define GPTConfig

config = GPTConfig(
    vocab_size=vocab_size,
    block_size=
)

## STEP 5: Defining Loss Function

In [ ]:
@torch.inference_model() # more optimized version of @torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item() # extract raw float
    model.train() # put back into training mode
    return out # return dict